In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/raw")
ELO_K = 20
ELO_INIT = 1500
ELO_HCA = 100

# Load data (mirrors 01_eda but focused on Elo inputs)
m_teams = pd.read_csv(DATA_DIR / "MTeams.csv")
w_teams = pd.read_csv(DATA_DIR / "WTeams.csv")
m_regular = pd.read_csv(DATA_DIR / "MRegularSeasonCompactResults.csv")
w_regular = pd.read_csv(DATA_DIR / "WRegularSeasonCompactResults.csv")
m_tourney = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
w_tourney = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")

print("✅ Data loaded for Elo computation")

✅ Data loaded for Elo computation


In [7]:
def run_elo(regular_df, tourney_df, k=ELO_K, init=ELO_INIT, hca=ELO_HCA):
    """Compute season-by-season Elo ratings for one gender.

    Returns a dict mapping (Season, TeamID) -> Elo.
    """
    elo = {}
    season_elos = {}
    all_games = pd.concat([regular_df, tourney_df]).sort_values(["Season", "DayNum"])  
    prev_season = None

    for _, row in all_games.iterrows():
        season = row["Season"]
        if season != prev_season and prev_season is not None:
            for tid, r in elo.items():
                season_elos[(prev_season, tid)] = r
            elo = {tid: 0.75 * r + 0.25 * init for tid, r in elo.items()}
        prev_season = season

        w_id, l_id = row["WTeamID"], row["LTeamID"]
        w_elo = elo.get(w_id, init)
        l_elo = elo.get(l_id, init)

        w_loc = row.get("WLoc", "N")
        w_adj = w_elo + (hca if w_loc == "H" else (-hca if w_loc == "A" else 0))

        exp_w = 1.0 / (1.0 + 10 ** ((l_elo - w_adj) / 400.0))
        elo[w_id] = w_elo + k * (1.0 - exp_w)
        elo[l_id] = l_elo + k * (0.0 - (1.0 - exp_w))

    if prev_season is not None:
        for tid, r in elo.items():
            season_elos[(prev_season, tid)] = r

    return season_elos

# Compute Elo for men and women
m_elos = run_elo(m_regular, m_tourney)
w_elos = run_elo(w_regular, w_tourney)

print(f"Men Elo entries:   {len(m_elos)}")
print(f"Women Elo entries: {len(w_elos)}")

Men Elo entries:   14206
Women Elo entries: 9952


In [8]:
# Show top teams for the latest season (roughly matching the ADK demo outputs)

m_names = dict(zip(m_teams["TeamID"], m_teams["TeamName"]))
w_names = dict(zip(w_teams["TeamID"], w_teams["TeamName"]))

latest_m = max(s for s, _ in m_elos.keys())
latest_w = max(s for s, _ in w_elos.keys())

top_m = sorted([(tid, r) for (s, tid), r in m_elos.items() if s == latest_m],
               key=lambda x: -x[1])[:5]
top_w = sorted([(tid, r) for (s, tid), r in w_elos.items() if s == latest_w],
               key=lambda x: -x[1])[:5]

print("Elo ratings computed.")
print("\nTop men's teams:")
for tid, r in top_m:
    print(f"- {m_names.get(tid, tid)}: {r:.0f}")

print("\nTop women's teams:")
for tid, r in top_w:
    print(f"- {w_names.get(tid, tid)}: {r:.0f}")

print(f"\nElo computed through {latest_m} (men) and {latest_w} (women).")

Elo ratings computed.

Top men's teams:
- Houston: 1821
- Duke: 1819
- Arizona: 1789
- Connecticut: 1785
- Gonzaga: 1756

Top women's teams:
- Connecticut: 1913
- South Carolina: 1893
- UCLA: 1869
- Texas: 1858
- LSU: 1807

Elo computed through 2026 (men) and 2026 (women).


In [9]:
# Aliases for older variable names used below
m_reg = m_regular
w_reg = w_regular
m_tour = m_tourney
w_tour = w_tourney

print("✅ Aliases set: m_reg/m_tour, w_reg/w_tour")

✅ Aliases set: m_reg/m_tour, w_reg/w_tour


In [10]:
import sys
import pandas as pd
from pathlib import Path

sys.path.append(str(Path("../src")))

from features import (
    normalize_games,
    build_team_season_stats,
    build_matchup_dataset,
)

In [11]:
m_reg_norm = normalize_games(m_reg, "M")
w_reg_norm = normalize_games(w_reg, "W")
m_tour_norm = normalize_games(m_tour, "M")
w_tour_norm = normalize_games(w_tour, "W")

all_games = pd.concat(
    [m_reg_norm, w_reg_norm, m_tour_norm, w_tour_norm],
    ignore_index=True
)

team_stats = build_team_season_stats(all_games)

print("Total games:", len(all_games))
print("Total team-seasons:", len(team_stats))
all_games.head()

print("Team1 win rate:", all_games["Team1Win"].mean())
print("PointDiff summary:")
print(all_games["PointDiff"].describe())

Total games: 341950
Total team-seasons: 23604
Team1 win rate: 0.49465418920894866
PointDiff summary:
count    341950.000000
mean         -0.181249
std          16.605840
min         -98.000000
25%         -11.000000
50%          -1.000000
75%          11.000000
max         108.000000
Name: PointDiff, dtype: float64


In [12]:
matchups = build_matchup_dataset(all_games, team_stats)

print("Matchup rows:", len(matchups))
matchups.head()

Matchup rows: 341950


,Season,Team1,Team2,WinPctDiff,AvgPDDiff,Team1Win
0,1985,1228,1328,-0.088235,3.280423,1
1,1985,1106,1354,0.041667,11.799242,1
2,1985,1112,1223,-0.037143,-5.538818,1
3,1985,1165,1432,0.021739,-6.257937,1
4,1985,1192,1447,0.321839,11.314103,1


In [13]:
print("WinPctDiff correlation:",
      matchups["WinPctDiff"].corr(matchups["Team1Win"]))

print("AvgPDDiff correlation:",
      matchups["AvgPDDiff"].corr(matchups["Team1Win"]))

WinPctDiff correlation: 0.5646668346047552
AvgPDDiff correlation: 0.5100809550122478


In [14]:
regular_season_games = pd.concat(
    [m_reg_norm, w_reg_norm],
    ignore_index=True
)

print("Regular season games:", len(regular_season_games))

team_stats_reg = build_team_season_stats(regular_season_games)

print("Team-seasons (regular only):", len(team_stats_reg))
team_stats_reg.head()

Regular season games: 337648
Team-seasons (regular only): 23604


,Season,TeamID,Games,Wins,AvgPointDiff,WinPct
0,1985,1102,24,5,-5.791667,0.208333
1,1985,1103,23,9,-3.043478,0.391304
2,1985,1104,30,21,7.800000,0.700000
3,1985,1106,24,10,-3.791667,0.416667
4,1985,1108,25,19,11.173913,0.760000


In [15]:
# Building Tournament Matchup Dataset
tournament_games = pd.concat(
    [m_tour_norm, w_tour_norm],
    ignore_index=True
)

print("Tournament games:", len(tournament_games))


tourney_matchups = build_matchup_dataset(
    tournament_games,
    team_stats_reg
)

print("Tournament matchup rows:", len(tourney_matchups))
tourney_matchups.head()

Tournament games: 4302
Tournament matchup rows: 4302


,Season,Team1,Team2,WinPctDiff,AvgPDDiff,Team1Win
0,1985,1116,1234,-0.030303,-8.377990,1
1,1985,1120,1345,-0.059310,-5.259615,1
2,1985,1207,1250,0.546616,20.939474,1
3,1985,1229,1425,0.062169,2.094474,1
4,1985,1242,1325,0.025926,2.217370,1


In [16]:
print(tourney_matchups[["WinPctDiff", "AvgPDDiff"]].corr())
print("Correlation with outcome:")
print(tourney_matchups.corr(numeric_only=True)["Team1Win"])

            WinPctDiff  AvgPDDiff
WinPctDiff    1.000000   0.744697
AvgPDDiff     0.744697   1.000000
Correlation with outcome:
Season       -0.021222
Team1         0.007852
Team2         0.009489
WinPctDiff    0.353434
AvgPDDiff     0.396708
Team1Win      1.000000
Name: Team1Win, dtype: float64
